# Phi-4 Multimodal — DIMER multi-capability tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/phi4-multimodal-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/phi4-multimodal-pipeline/blob/main/tutorials/phi4_multimodal_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2FPhi--4--multimodal--instruct-ffcc4d?style=flat)](https://huggingface.co/microsoft/Phi-4-multimodal-instruct) [![arXiv](https://img.shields.io/badge/arXiv-2503.01743-b31b1b.svg)](https://arxiv.org/abs/2503.01743)

**Profile:** `MULTI-CAPABILITY`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** text-, image-, and audio-conditioned text generation using one pinned `microsoft/Phi-4-multimodal-instruct` checkpoint

**This notebook is standalone.** It carries the repository's pipeline module (`src/phi4_multimodal_pipeline/pipeline.py` at revision `0c8003a2a495`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `93f923e1a7727d1c4f446756212d9d3e8fcc5d81` (~11173 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference the 5.6B language model consumes a chat prompt in which `<|image_k|>` and `<|audio_k|>` placeholders are replaced by SigLIP vision embeddings and conformer speech embeddings (each modality routed through its own LoRA adapter inside the checkpoint), then generates text autoregressively; the pipeline builds that prompt, runs greedy decoding by default and returns the generated text with provenance. **No adaptation occurs:** no training, fine-tuning, in-context conditioning beyond the single instruction, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, the processor configuration **and the model code**, and the carried pipeline module adds snapshot verification, the explicit remote-code opt-in, the input contract, a fixed output contract and the `validate_inputs` and `evaluation_report` helpers. **Trust boundary (MOD9/MOD10):** this release ships its model, configuration and processor as Python files that `transformers` must execute (`trust_remote_code=True` inside the carried loader, gated behind `allow_remote_code=True`); the standalone path pins each of those files by SHA-256 in the inline manifest, fetches them at the immutable revision and re-hashes them before they are imported, so the code that runs is exactly the reviewed revision's — no other remote code is executed. The default samples are a text instruction, a synthetic traffic-sign image drawn in code, and the checkpoint's own example speech clip; their outputs are demonstration (plumbing) evidence, not a benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream revision including its remote model code, acknowledge the custom-code trust boundary explicitly, validate each capability's input into an input manifest, run text-only, image + text and audio + text generation through one public API, exercise optional BYOD image and audio inputs, produce an evaluation report that is honestly `not-measurable` for open-ended generation, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** fine-tuning (the checkpoint's `sample_finetune_*.py` scripts are pinned but never run), FlashAttention 2 (available only by explicit caller choice with a compatible `flash-attn` build), video, tool use, multi-turn chat state, calibrated answer confidence, or any claim that upstream benchmark results were reproduced. Generated descriptions and transcripts can be wrong and the pipeline does not detect it.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12) with a **CUDA GPU** and roughly 16 GB of GPU memory for the ~11.2 GB snapshot loaded in its stored bfloat16 precision; `from_pretrained` refuses `device='cuda'` without a CUDA device. Attention is `eager` for portability. The pinned `torch==2.6.0` install and the three SafeTensors shards are the largest downloads of the run.
- **Knowledge:** basic Python and PIL; what a chat prompt, greedy decoding and `trust_remote_code` mean.
- **Data:** the default image is a synthetic red octagon with a white border drawn in code (no download, no ground truth); the default audio is `examples/what_is_the_traffic_sign_in_the_image.wav`, a short speech clip that ships inside the pinned checkpoint snapshot itself and is therefore fetched from the Hugging Face Hub at the immutable revision and digest-verified with the weights. Optional BYOD uploads are gated off by default so the sample path can run top-to-bottom without interaction; expected BYOD inputs are one image file decodable by Pillow and/or one audio file decodable by `soundfile`. Do not upload confidential or restricted media to a hosted notebook environment unless you are authorized to do so. Inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/Phi-4-multimodal-instruct` snapshot (~11173 MB in total) at revision `93f923e1a772…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.6.0',
    'torchvision==0.21.0',
    'transformers==4.48.2',
    'accelerate==1.3.0',
    'huggingface-hub==0.36.2',
    'soundfile==0.13.1',
    'pillow==11.1.0',
    'scipy==1.15.2',
    'backoff==2.2.1',
    'peft==0.13.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'phi4-multimodal-pipeline',
    'repository_revision': '0c8003a2a495377acc8cfd8eadd62cf8a00d3bd3',
    'embedded_module': 'src/phi4_multimodal_pipeline/pipeline.py',
    'embedded_modules': ['src/phi4_multimodal_pipeline/pipeline.py'],
    'module_sha256': '1008eb0d17b31c7420880187345b7beb69e220cad151c5c5140383222dbce150',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/phi4_multimodal_pipeline/` @ `0c8003a2a495`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/phi4_multimodal_pipeline/pipeline.py`

In [ ]:
"""Text-, image- and audio-conditioned generation with the pinned ``microsoft/Phi-4-multimodal-instruct``.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision. **Trust boundary
(MOD9/MOD10):** this upstream release ships its model, configuration and processor as custom Python
files (``configuration_phi4mm.py``, ``modeling_phi4mm.py``, ``processing_phi4mm.py``,
``speech_conformer_encoder.py``, ``vision_siglip_navit.py``) that transformers must execute, so the
loader passes ``trust_remote_code=True``. It refuses to do so unless the caller opts in with
``allow_remote_code=True``, and on the snapshot path every one of those files is pinned by SHA-256 in
the manifest and re-hashed by ``verify_snapshot`` before it is imported; the weights are SafeTensors.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"
MODEL_REVISION = "93f923e1a7727d1c4f446756212d9d3e8fcc5d81"
MODEL_LICENSE = "MIT"
MODEL_KEY = "phi4-multimodal-instruct"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
CONFIG_FILE = "config.json"
# Upstream Python files that transformers executes when trust_remote_code=True (pinned by digest).
REMOTE_CODE_FILES = (
    "configuration_phi4mm.py",
    "modeling_phi4mm.py",
    "processing_phi4mm.py",
    "speech_conformer_encoder.py",
    "vision_siglip_navit.py",
)
# MOD9/MOD10: why the loader enables remote code, and what bounds it (the fleet card gate keys on this name).
REMOTE_CODE_JUSTIFICATION = (
    "upstream ships Phi4MMForCausalLM / Phi4MMProcessor / Phi4MMConfig only as custom Python files in the "
    "model repository (config.json auto_map); transformers==4.48.2 has no built-in implementation, so "
    "trust_remote_code=True is unavoidable. Bounded by: the explicit allow_remote_code=True opt-in, the "
    "immutable MODEL_REVISION on every path, and on the snapshot path the per-file SHA-256 pins of "
    "REMOTE_CODE_FILES re-hashed by verify_snapshot before transformers imports them."
)
SUPPORTED_ATTENTION = {"eager", "flash_attention_2"}
MAX_IMAGES = 4  # images per request on the DIMER reference path
MAX_AUDIOS = 4  # audio clips per request on the DIMER reference path
MAX_NEW_TOKENS = 2048
MAX_TEMPERATURE = 2.0
DEFAULT_MAX_NEW_TOKENS = 128
DEFAULT_TEMPERATURE = 0.0  # greedy decoding


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    listed = {entry["path"] for entry in manifest.get("files", [])}
    missing_code = [name for name in REMOTE_CODE_FILES if name not in listed]
    if missing_code:
        raise ValueError(f"manifest does not pin the remote-code files {missing_code}; refusing")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest and the
    small config/tokenizer files but git-ignores the shards, the remote code and the sample media).
    Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _build_prompt(instruction: str, image_count: int, audio_count: int) -> str:
    if not instruction or not instruction.strip():
        raise ValueError("instruction must be non-empty")
    if image_count > MAX_IMAGES or audio_count > MAX_AUDIOS:
        raise ValueError(
            "DIMER reference path allows at most 4 images and 4 audio clips per request"
        )
    media = "".join(f"<|image_{index}|>" for index in range(1, image_count + 1))
    media += "".join(f"<|audio_{index}|>" for index in range(1, audio_count + 1))
    return f"<|user|>{media}{instruction.strip()}<|end|><|assistant|>"


INPUT_SCHEMA: dict[str, Any] = {
    "instruction": "non-empty text; the user turn of the chat template",
    "images": f"optional sequence of PIL images, at most {MAX_IMAGES}",
    "audios": f"optional sequence of (float waveform, sampling_rate) tuples, at most {MAX_AUDIOS}",
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "temperature": [0.0, MAX_TEMPERATURE],
    "decoding": "temperature 0 → greedy (do_sample=False); otherwise sampling at that temperature",
    "preprocessing": (
        "the upstream processor builds the <|user|>…<|end|><|assistant|> prompt, tiles images for the SigLIP "
        "encoder and computes speech features; nothing is altered by this module"
    ),
}


def _check_inputs(
    instruction: str,
    images: Sequence[Any] | None,
    audios: Sequence[Any] | None,
    max_new_tokens: int,
    temperature: float,
) -> tuple[list[Any], list[Any], str]:
    """Raise ValueError naming the first violated ceiling; return the media lists and the prompt."""
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError("max_new_tokens must be between 1 and 2048")
    if not 0.0 <= temperature <= MAX_TEMPERATURE:
        raise ValueError("temperature must be between 0 and 2")
    image_list = list(images or [])
    audio_list = list(audios or [])
    prompt = _build_prompt(instruction, len(image_list), len(audio_list))
    return image_list, audio_list, prompt


def _observe_image(image: Any) -> dict[str, Any]:
    size = getattr(image, "size", None)
    return {"kind": "image", "mode": getattr(image, "mode", None), "size": list(size) if size else None}


def _observe_audio(audio: Any) -> dict[str, Any]:
    if isinstance(audio, tuple | list) and len(audio) == 2:
        waveform, rate = audio
        samples = len(waveform) if hasattr(waveform, "__len__") else None
        has_rate = isinstance(rate, int | float) and bool(rate)
        seconds = round(samples / rate, 3) if samples is not None and has_rate else None
        return {"kind": "audio", "samples": samples, "sampling_rate": rate, "seconds": seconds}
    return {"kind": "audio", "samples": None, "sampling_rate": None, "seconds": None}


def validate_inputs(
    instruction: str,
    *,
    images: Sequence[Any] | None = None,
    audios: Sequence[Any] | None = None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observed media, request, verdict).

    Rejection is reported by raising exactly as ``generate`` would; a caller that wants the
    finding recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    image_list, audio_list, prompt = _check_inputs(
        instruction, images, audios, max_new_tokens, temperature
    )
    observed = [_observe_image(image) for image in image_list]
    observed += [_observe_audio(audio) for audio in audio_list]
    if names is not None and len(names) != len(observed):
        raise ValueError("names must have one entry per media input")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"{item['kind']}-{i}", **item} for i, item in enumerate(observed)
        ],
        "instruction_chars": len(instruction.strip()),
        "prompt_chars": len(prompt),
        "image_count": len(image_list),
        "audio_count": len(audio_list),
        "max_new_tokens": max_new_tokens,
        "temperature": temperature,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Mapping[str, Mapping[str, Any]], *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though nothing is measurable.

    ``results`` maps a capability label (``text``, ``image``, ``audio``) to a ``generate`` result. The
    repository ships no metric helper for open-ended generation, so the verdict is always
    ``not-measurable`` (EVAL9) and the report states, per capability, what labelled data would make it
    measurable; the generated texts are sanity evidence that each path executed.
    """
    needs = {
        "text": "reference answers and a task metric (exact match or a rubric) on instruction-answer pairs",
        "image": "image-question pairs with reference answers (VQA accuracy) or reference captions (CIDEr)",
        "audio": "reference transcripts scored with a word error rate after a stated normalisation",
    }
    capabilities = []
    for label, result in results.items():
        text = str(result.get("text", ""))
        capabilities.append(
            {
                "capability": label,
                "generated_chars": len(text),
                "non_empty": bool(text.strip()),
                "image_count": int(result.get("image_count", 0)),
                "audio_count": int(result.get("audio_count", 0)),
                "temperature": result.get("temperature"),
                "needs": needs.get(label, "labelled references for this capability"),
            }
        )
    return {
        "task": "multimodal instruction-following text generation",
        "score_semantics": "generated text; the pipeline exposes no likelihood, confidence or threshold",
        "sample_kind": sample_kind,
        "n_capabilities": len(capabilities),
        "capabilities": capabilities,
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "no reference answers, captions or transcripts were supplied and the repository ships no metric "
            "helper for open-ended generation; non-empty output only shows that each capability path executed"
        ),
        "needs": "; ".join(f"{c['capability']}: {c['needs']}" for c in capabilities) or "labelled references",
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class Phi4MultimodalPipeline:
    _generate: Callable[..., str]
    device: str
    attention_implementation: str = "eager"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        *,
        allow_remote_code: bool = False,
        device: str = "cuda",
        attention_implementation: str = "eager",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Phi4MultimodalPipeline:
        if not allow_remote_code:
            raise RuntimeError(
                "Phi-4-multimodal requires upstream custom Python model code. "
                "Set allow_remote_code=True only after reviewing the pinned revision trust boundary."
            )
        if attention_implementation not in SUPPORTED_ATTENTION:
            raise ValueError(
                "attention_implementation must be 'eager' or 'flash_attention_2'"
            )

        import torch
        from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig

        if device.startswith("cuda") and not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA was requested but is unavailable; choose an appropriate GPU runtime"
            )
        if attention_implementation == "flash_attention_2":
            try:
                import flash_attn  # noqa: F401
            except ImportError as exc:
                raise RuntimeError(
                    "flash_attention_2 was requested but flash-attn is not installed; "
                    "install a compatible flash-attn build or use attention_implementation='eager'"
                ) from exc

        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            # A directory argument makes transformers read the config, the (digest-verified) remote
            # code, the tokenizer and the SafeTensors shards from it directly — no Hub resolution.
            location: dict[str, Any] = {"pretrained_model_name_or_path": str(root)}
            source = "local-snapshot"
        elif allow_download:
            location = {"pretrained_model_name_or_path": MODEL_ID, "revision": MODEL_REVISION}
            source = "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )

        processor = AutoProcessor.from_pretrained(**location, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            **location,
            trust_remote_code=True,
            torch_dtype="auto",
            device_map=device,
            _attn_implementation=attention_implementation,
        ).eval()
        generation_config = GenerationConfig.from_pretrained(**location)

        def runner(
            prompt: str,
            images: Sequence[Any] | None,
            audios: Sequence[Any] | None,
            max_new_tokens: int,
            temperature: float,
        ) -> str:
            processor_kwargs: dict[str, Any] = {
                "text": prompt,
                "return_tensors": "pt",
            }
            if images:
                processor_kwargs["images"] = list(images)
            if audios:
                processor_kwargs["audios"] = list(audios)
            inputs = processor(**processor_kwargs).to(model.device)

            do_sample = temperature > 0
            generation_kwargs: dict[str, Any] = {
                "max_new_tokens": max_new_tokens,
                "do_sample": do_sample,
                "generation_config": generation_config,
            }
            if do_sample:
                generation_kwargs["temperature"] = temperature
            generated = model.generate(**inputs, **generation_kwargs)
            generated = generated[:, inputs["input_ids"].shape[1] :]
            return processor.batch_decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )[0].strip()

        return cls(runner, device, attention_implementation, source)

    def generate(
        self,
        instruction: str,
        *,
        images: Sequence[Any] | None = None,
        audios: Sequence[Any] | None = None,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        temperature: float = DEFAULT_TEMPERATURE,
    ) -> dict[str, Any]:
        image_list, audio_list, prompt = _check_inputs(
            instruction, images, audios, max_new_tokens, temperature
        )
        text = self._generate(
            prompt,
            image_list or None,
            audio_list or None,
            max_new_tokens,
            temperature,
        )
        return {
            "text": text,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "image_count": len(image_list),
            "audio_count": len(audio_list),
            "temperature": temperature,
            "max_new_tokens": max_new_tokens,
            "attention_implementation": self.attention_implementation,
            "device": self.device,
            "source": self.source,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `26`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `93f923e1a772…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Phi4MultimodalPipeline.from_pretrained(allow_remote_code=True, weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "phi4-multimodal-instruct",
  "modelId": "microsoft/Phi-4-multimodal-instruct",
  "revision": "93f923e1a7727d1c4f446756212d9d3e8fcc5d81",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 1141,
      "sha256": "c2cfccb812fe482101a8f04597dfc5a9991a6b2748266c47ac91b6a5aae15383"
    },
    {
      "path": "README.md",
      "bytes": 65395,
      "sha256": "c79dc706ea8478e931981a048164233ae2256b229114e05744610b87f9301293"
    },
    {
      "path": "added_tokens.json",
      "bytes": 249,
      "sha256": "d4f2aceb0f20b71dd1f4bcc7e052e4412946bf281840b8f83d39f259571af486"
    },
    {
      "path": "config.json",
      "bytes": 4631,
      "sha256": "49e1c05f93d43d7f17715b779a2576235b019f587285d7d914e5b05156253f62"
    },
    {
      "path": "configuration_phi4mm.py",
      "bytes": 11014,
      "sha256": "bd9609bd47ba0c87788011e5158a8bd3e1e93165a82a9b764eb7cb048006c949"
    },
    {
      "path": "examples/what_is_shown_in_this_image.wav",
      "bytes": 112844,
      "sha256": "9fcb7788d94740c7925b055a6e66d3bae6e2344ac5caa6ba47da715a3249ef66"
    },
    {
      "path": "examples/what_is_the_traffic_sign_in_the_image.wav",
      "bytes": 741454,
      "sha256": "16faff68324cebb79626268af7eb4e2796f6fafd60cfa0ed2bbef927b7d8188a"
    },
    {
      "path": "generation_config.json",
      "bytes": 190,
      "sha256": "757daa0d0e89171fe48fc3286341833e95b90bdb7dd3b02a2f8920fb09f85a38"
    },
    {
      "path": "merges.txt",
      "bytes": 2418348,
      "sha256": "856ce61180bb689282eed6b3a6838bb1f438399be23aefe9d20eb379791fb4ad"
    },
    {
      "path": "model-00001-of-00003.safetensors",
      "bytes": 4997504848,
      "sha256": "c46bb03332d82f6a3eaf85bd20af388dd4d4d68b198c2203c965c7381a466094"
    },
    {
      "path": "model-00002-of-00003.safetensors",
      "bytes": 4952333128,
      "sha256": "b3e812c0c8acef4e7f5e34d6c9f77a7640ee4a2b93ea351921365ac62f19918d"
    },
    {
      "path": "model-00003-of-00003.safetensors",
      "bytes": 1199389232,
      "sha256": "7be96b7339303752634b202d3f377bcf312a03046586eca6cea23347ace1e65a"
    },
    {
      "path": "model.safetensors.index.json",
      "bytes": 239890,
      "sha256": "b67dbc7062e1ccf472faba4222d631dc42929c827fbdaed1ec8e34fe0601819a"
    },
    {
      "path": "modeling_phi4mm.py",
      "bytes": 116057,
      "sha256": "e2b44eb7a66d6cc54524cee1ff9ba92d0658d435ea8900329ea0dbdb85c6439d"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 482,
      "sha256": "9db19b9663fb86f04c0f11d3a9b7f65a19f13d4543fb4a15bd33f82b0c92d64f"
    },
    {
      "path": "processing_phi4mm.py",
      "bytes": 32775,
      "sha256": "84914d3e12256b4e2186e040c9830c11408468b6774f42afe85e6f8de2626d50"
    },
    {
      "path": "processor_config.json",
      "bytes": 121,
      "sha256": "798fc4cd09c067053af27f07f0d2b329b471c5b2eb923ceb06efa41dee660c05"
    },
    {
      "path": "sample_finetune_speech.py",
      "bytes": 16669,
      "sha256": "0f2d44593bd0142d3f985473b90d8baf884973fe4d55f139a9a035e333ecc9be"
    },
    {
      "path": "sample_finetune_vision.py",
      "bytes": 19617,
      "sha256": "dee385e684dcab9c411cef87d41a4fcd897cce528712a23a9c00fd008eb8de3e"
    },
    {
      "path": "sample_inference_phi4mm.py",
      "bytes": 10498,
      "sha256": "22bfc76ecdaeb2641bc6089a8efcaf2291e017111f0b6db3be91df451adab3a8"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 473,
      "sha256": "57491904f8680d4b52ed440f1f7ba48cad1c31ecf3eb453b03484e6ff4723ae8"
    },
    {
      "path": "speech_conformer_encoder.py",
      "bytes": 110521,
      "sha256": "3742827e945732cc5deea4a95e14004da037044431a94e3f3fac26239e614e3a"
    },
    {
      "path": "tokenizer.json",
      "bytes": 15524479,
      "sha256": "4c1b9f641d4f8b7247b8d5007dd3b6a9f6a87cb5123134fe0d326f14d10c0585"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 3248,
      "sha256": "a5da2e45718db78924ad5135a58a80b0303596acf54a1dc5c912c98436ddcaf3"
    },
    {
      "path": "vision_siglip_navit.py",
      "bytes": 78218,
      "sha256": "7d5c053341ee9c099126fe675d5dcdc0ed5c0246f92fffdec78a1ab2f804e28d"
    },
    {
      "path": "vocab.json",
      "bytes": 3910310,
      "sha256": "6cb65a857824fa6615bb1782d95d882617a8bbce1da0317118586b36f39e98bd"
    }
  ],
  "totalBytes": 11172645832
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Phi4MultimodalPipeline.from_pretrained(allow_remote_code=True, weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Prepare the samples or optional BYOD

Three inputs, one per capability. **Text:** a fixed instruction. **Image:** a synthetic 256×256 traffic-sign-like image drawn in code — a red octagon with a white border on a light background, deterministic, digest printed; it is not a photograph, so the model's description is a sanity check of the vision path, not a correctness measurement. **Audio:** the checkpoint's own `examples/what_is_the_traffic_sign_in_the_image.wav` (already staged and digest-verified in Section 3 because it is a manifest entry), decoded with the pinned `soundfile` dependency into a float32 waveform plus sampling rate and passed as the `(array, sampling_rate)` tuple the upstream processor expects. BYOD is optional and disabled by default; `USE_BYOD` enables upload dialogs for an image and/or an audio file — skip either dialog by cancelling it. Look for a dictionary naming each sample's kind and digest.

In [ ]:
import hashlib
import io

import numpy as np
import soundfile as sf
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
TEXT_INSTRUCTION = 'In two sentences, explain what automatic speech recognition does.'
IMAGE_INSTRUCTION = 'Describe the most prominent traffic sign in the image.'
AUDIO_INSTRUCTION = 'Transcribe the attached speech.'
SAMPLE_AUDIO = WEIGHTS_DIR / 'examples' / 'what_is_the_traffic_sign_in_the_image.wav'

def _synthetic_sign(side=256):
    # Deterministic red octagon with a white border on a light grey background: no randomness, stable digest.
    canvas = Image.new('RGB', (side, side), (235, 235, 235))
    draw = ImageDraw.Draw(canvas)
    centre, radius = side / 2, side * 0.42
    octagon = [(centre + radius * np.cos(np.pi / 8 + k * np.pi / 4), centre + radius * np.sin(np.pi / 8 + k * np.pi / 4)) for k in range(8)]
    draw.polygon(octagon, fill=(200, 16, 24), outline=(255, 255, 255), width=max(2, side // 40))
    return canvas

if USE_BYOD:
    from google.colab import files
    print('Upload an image (or cancel to keep the synthetic sign).')
    uploaded = files.upload()
    if uploaded:
        image_name, image_bytes = next(iter(uploaded.items()))
        image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        image_kind = 'BYOD'
    else:
        image, image_name, image_kind = _synthetic_sign(), 'synthetic_octagon_256.png', 'synthetic'
    print('Upload an audio file (or cancel to keep the checkpoint example clip).')
    uploaded = files.upload()
    if uploaded:
        audio_name, audio_bytes = next(iter(uploaded.items()))
        waveform, sampling_rate = sf.read(io.BytesIO(audio_bytes), dtype='float32')
        audio_kind = 'BYOD'
    else:
        waveform, sampling_rate = sf.read(SAMPLE_AUDIO, dtype='float32')
        audio_name, audio_kind = SAMPLE_AUDIO.name, 'checkpoint-example'
else:
    image, image_name, image_kind = _synthetic_sign(), 'synthetic_octagon_256.png', 'synthetic'
    waveform, sampling_rate = sf.read(SAMPLE_AUDIO, dtype='float32')
    audio_name, audio_kind = SAMPLE_AUDIO.name, 'checkpoint-example'
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1)
audio = (waveform, sampling_rate)
image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
audio_sha256 = hashlib.sha256(np.ascontiguousarray(waveform, dtype=np.float32).tobytes()).hexdigest()
sample_kind = 'BYOD' if USE_BYOD else 'synthetic'
print({'image': {'kind': image_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256}, 'audio': {'kind': audio_kind, 'name': audio_name, 'seconds': round(len(waveform) / sampling_rate, 2), 'sampling_rate': sampling_rate, 'waveform_sha256': audio_sha256}})

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `generate` applies — a non-empty instruction, at most `MAX_IMAGES` images and `MAX_AUDIOS` audio clips, `max_new_tokens` 1..`MAX_NEW_TOKENS`, `temperature` 0..`MAX_TEMPERATURE` — and returns an **input manifest** naming the schema and ceilings, each media input's observed properties (image mode and size; audio samples, sampling rate, duration), the request (prompt length, decoding settings) and the verdict. One manifest is produced per capability and the three are written together to `outputs/phi4_multimodal_input_manifest.json`. To show what rejection looks like, the cell also validates a request with five images and records the pipeline's own error message as a finding. Inside the upstream processor images are tiled for the SigLIP encoder and audio is converted to speech features; nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_IMAGES': MAX_IMAGES, 'MAX_AUDIOS': MAX_AUDIOS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_TEMPERATURE': MAX_TEMPERATURE}})
input_manifest = {
    'text': validate_inputs(TEXT_INSTRUCTION, max_new_tokens=96),
    'image': validate_inputs(IMAGE_INSTRUCTION, images=[image], max_new_tokens=96, names=[image_name]),
    'audio': validate_inputs(AUDIO_INSTRUCTION, audios=[audio], max_new_tokens=128, names=[audio_name]),
}
# Demonstrate rejection on a request that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(IMAGE_INSTRUCTION, images=[image] * (MAX_IMAGES + 1))
except ValueError as exc:
    input_manifest['image']['findings'].append({'input': 'too-many-images-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/phi4_multimodal_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Capability A — text-only generation

**Input contract:** one instruction string, no media. **Output contract:** `text` is the decoded continuation after the `<|assistant|>` tag, greedy (`temperature=0.0` → `do_sample=False`) and capped at `max_new_tokens`; the result also echoes the media counts, decoding settings, attention backend, device and weight source. The text is generated language, not a calibrated statement — a fluent answer is not evidence of correctness.

In [ ]:
text_result = pipe.generate(TEXT_INSTRUCTION, max_new_tokens=96, temperature=0.0)
print({'image_count': text_result['image_count'], 'audio_count': text_result['audio_count'], 'temperature': text_result['temperature'], 'attention_implementation': text_result['attention_implementation'], 'device': text_result['device'], 'source': text_result['source']})
print(text_result['text'])

## 7. Capability B — image + text

**Input contract:** one instruction plus up to `MAX_IMAGES` PIL images; the pipeline inserts one `<|image_k|>` placeholder per image and the upstream processor tiles each image for the SigLIP encoder. **Output contract:** generated text describing or answering about the images. On the synthetic octagon the description is a sanity check that the vision path executes — the image has no ground-truth caption, and a plausible "stop sign" answer must not be read as recognition accuracy.

In [ ]:
image_result = pipe.generate(IMAGE_INSTRUCTION, images=[image], max_new_tokens=96, temperature=0.0)
print({'image_count': image_result['image_count'], 'audio_count': image_result['audio_count']})
print(image_result['text'])

## 8. Capability C — audio + text

**Input contract:** one instruction plus up to `MAX_AUDIOS` `(waveform, sampling_rate)` tuples; the pipeline inserts one `<|audio_k|>` placeholder per clip and the upstream processor computes speech features. **Output contract:** generated text — here a transcription request. The checkpoint's example clip is a short spoken question; its file name suggests the expected words, but that is not a reference transcript and no accuracy is inferred from this one clip.

In [ ]:
audio_result = pipe.generate(AUDIO_INSTRUCTION, audios=[audio], max_new_tokens=128, temperature=0.0)
print({'image_count': audio_result['image_count'], 'audio_count': audio_result['audio_count']})
print(audio_result['text'])

## 9. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. The repository ships **no metric helper** for open-ended generation, so the verdict is `not-measurable` by construction: the report records, per capability, whether the output was non-empty (plumbing evidence only) and what labelled data would make it measurable — reference answers with a task metric for text, image-question pairs with reference answers (VQA accuracy) or captions (CIDEr) for images, and reference transcripts scored with a word error rate for audio. The report is written to `outputs/phi4_multimodal_evaluation_report.json`. A `sample-sanity` verdict is deliberately not available here because no metric exists in the repository to back it.

In [ ]:
results = {'text': text_result, 'image': image_result, 'audio': audio_result}
report = evaluation_report(results, sample_kind=sample_kind)
with open('outputs/phi4_multimodal_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
print('No reference answers, captions or transcripts exist for these samples; the outputs above are sanity evidence only.')

## 10. Export outputs and provenance

Machine-readable JSON preserves each capability's full result, the evaluation report, the input manifests, the sample identities and digests, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision (which pins the remote code as well as the weights), the model licence, the acknowledged trust boundary, and the runtime identity (Python, `torch`, `transformers`, device, attention backend). The three generated texts are also written as one plain-text file. No credentials are recorded.

In [ ]:
payload = {
    'capabilities': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'samples': {'text': {'instruction': TEXT_INSTRUCTION}, 'image': {'kind': image_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256}, 'audio': {'kind': audio_kind, 'name': audio_name, 'sampling_rate': sampling_rate, 'waveform_sha256': audio_sha256}},
    'trust_boundary': {'remote_code_executed': True, 'remote_code_files': list(REMOTE_CODE_FILES), 'pinned_by': 'inline manifest SHA-256 at MODEL_REVISION, verified by verify_snapshot before import', 'opt_in': 'allow_remote_code=True'},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'attention_implementation': pipe.attention_implementation,
    },
}
with open('outputs/phi4_multimodal_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/phi4_multimodal_generations.txt', 'w', encoding='utf-8') as handle:
    for label, result in results.items():
        handle.write(f'[{label}]\n{result["text"]}\n\n')
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The three outputs share one generative model but have different evidence sources and failure modes. Non-empty text only establishes that the inference path executed; the evaluation report is `not-measurable` because the repository ships no metric and the samples carry no references. Image and audio interpretations can be wrong, the synthetic octagon is not a photograph of any sign, the example clip has no reference transcript, and no calibrated answer confidence is returned. The model executes upstream Python code: that code is pinned by digest at the immutable revision and re-hashed before import, which fixes *which* code runs but does not make it audited — review the pinned files before any deployment. The tutorial deliberately does not expose fine-tuning or claim that upstream benchmark results were reproduced.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model **and its remote code**, execute the explicitly acknowledged custom-code boundary, validate the demonstrated inputs, run the demonstrated public pipeline capabilities, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a photograph of a real traffic sign and a recording you have transcribed yourself; pass both media in one call (`images=[image], audios=[audio]`) with the checkpoint's other example clip `what_is_shown_in_this_image.wav` to reproduce the upstream image-question demo; raise `temperature` above 0 and observe non-deterministic sampling; compare `attention_implementation='flash_attention_2'` only after installing a compatible `flash-attn` build.

## References

- Repository README: https://github.com/kurtvalcorza/phi4-multimodal-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/phi4-multimodal-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/phi4-multimodal-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/Phi-4-multimodal-instruct
- Pinned upstream sample code: https://huggingface.co/microsoft/Phi-4-multimodal-instruct/blob/93f923e1a7727d1c4f446756212d9d3e8fcc5d81/sample_inference_phi4mm.py
- Technical report: https://arxiv.org/abs/2503.01743